In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

In [10]:
# === Load dataset ===
df = pd.read_csv("3Types-bottleneck-dataset.csv")

# === Drop timestamp ===
df.drop(columns=["timestamp"], inplace=True)

# === Normalize label column (lowercase all labels) ===
df["label"] = df["label"].str.strip().str.lower()

df.info()
df['label'].value_counts()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2476 entries, 0 to 2475
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   signal_strength_percent      2476 non-null   float64
 1   channel_congestion_percent   2476 non-null   float64
 2   gateway_ping_ms              2476 non-null   float64
 3   gateway_packet_loss_percent  2476 non-null   float64
 4   crc_error_rate               2476 non-null   float64
 5   label                        2476 non-null   object 
dtypes: float64(5), object(1)
memory usage: 116.2+ KB


,signal_strength_percent,channel_congestion_percent,gateway_ping_ms,gateway_packet_loss_percent,crc_error_rate
count,2476.000000,2476.000000,2476.000000,2476.000000,2476.000000
mean,-0.116988,-0.024630,0.927909,0.425733,0.192934
std,1.065203,1.410982,2.313223,3.968043,1.004631
min,-1.000000,-1.000000,0.000000,0.000000,0.000000
25%,-1.000000,-1.000000,0.188920,0.000000,0.000000
50%,-1.000000,-1.000000,0.283000,0.000000,0.000000
75%,1.000000,0.877415,0.576963,0.000000,0.000000
max,3.995266,9.886334,49.220763,66.670000,9.873962


In [11]:
# === Encode target labels ===
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
joblib.dump(label_encoder, 'label_encoder3Types.pkl')
X = df.drop(columns=["label", "label_encoded"])
y = df["label_encoded"]

In [12]:
# === Train-test split (Stratified) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(xgb.__version__)

3.0.2


In [13]:
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    eval_metric="mlogloss",
    use_label_encoder=False,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    #subsample=0.8,
    #colsample_bytree=0.8,
    random_state=42,
    tree_method='exact'
)

# Train without early stopping
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

[0]	validation_0-mlogloss:1.50862
[1]	validation_0-mlogloss:1.29903
[2]	validation_0-mlogloss:1.13352
[3]	validation_0-mlogloss:0.99783
[4]	validation_0-mlogloss:0.88357
[5]	validation_0-mlogloss:0.78605
[6]	validation_0-mlogloss:0.70203
[7]	validation_0-mlogloss:0.62886
[8]	validation_0-mlogloss:0.56473
[9]	validation_0-mlogloss:0.50823
[10]	validation_0-mlogloss:0.45815
[11]	validation_0-mlogloss:0.41373
[12]	validation_0-mlogloss:0.37410
[13]	validation_0-mlogloss:0.33881
[14]	validation_0-mlogloss:0.30720
[15]	validation_0-mlogloss:0.27897
[16]	validation_0-mlogloss:0.25366
[17]	validation_0-mlogloss:0.23091
[18]	validation_0-mlogloss:0.21057
[19]	validation_0-mlogloss:0.19228
[20]	validation_0-mlogloss:0.17584
[21]	validation_0-mlogloss:0.16105
[22]	validation_0-mlogloss:0.14781
[23]	validation_0-mlogloss:0.13587
[24]	validation_0-mlogloss:0.12513
[25]	validation_0-mlogloss:0.11544
[26]	validation_0-mlogloss:0.10675
[27]	validation_0-mlogloss:0.09889
[28]	validation_0-mlogloss:0.0

c:\Users\msaad\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:16:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[67]	validation_0-mlogloss:0.02821
[68]	validation_0-mlogloss:0.02814
[69]	validation_0-mlogloss:0.02806
[70]	validation_0-mlogloss:0.02801
[71]	validation_0-mlogloss:0.02796
[72]	validation_0-mlogloss:0.02792
[73]	validation_0-mlogloss:0.02802
[74]	validation_0-mlogloss:0.02795
[75]	validation_0-mlogloss:0.02791
[76]	validation_0-mlogloss:0.02786
[77]	validation_0-mlogloss:0.02777
[78]	validation_0-mlogloss:0.02774
[79]	validation_0-mlogloss:0.02766
[80]	validation_0-mlogloss:0.02763
[81]	validation_0-mlogloss:0.02758
[82]	validation_0-mlogloss:0.02754
[83]	validation_0-mlogloss:0.02750
[84]	validation_0-mlogloss:0.02749
[85]	validation_0-mlogloss:0.02752
[86]	validation_0-mlogloss:0.02747
[87]	validation_0-mlogloss:0.02748
[88]	validation_0-mlogloss:0.02744
[89]	validation_0-mlogloss:0.02747
[90]	validation_0-mlogloss:0.02741
[91]	validation_0-mlogloss:0.02741
[92]	validation_0-mlogloss:0.02737
[93]	validation_0-mlogloss:0.02733
[94]	validation_0-mlogloss:0.02728
[95]	validation_0-ml

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [14]:
# === Predict and evaluate ===
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)  # Confidence scores

# === Display results ===
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
                precision    recall  f1-score   support

    congestion       1.00      1.00      1.00        10
           crc       1.00      1.00      1.00        10
   gatewayloss       1.00      1.00      1.00         5
   gatewayping       0.95      0.95      0.95        19
        normal       1.00      1.00      1.00       442
signalstrength       1.00      1.00      1.00        10

      accuracy                           1.00       496
     macro avg       0.99      0.99      0.99       496
  weighted avg       1.00      1.00      1.00       496

Confusion Matrix:
[[ 10   0   0   0   0   0]
 [  0  10   0   0   0   0]
 [  0   0   5   0   0   0]
 [  0   0   0  18   1   0]
 [  0   0   0   1 441   0]
 [  0   0   0   0   0  10]]


In [15]:
# === Example: Print prediction with confidence ===
for i in range(5):
    probs = y_proba[i]
    pred_class = np.argmax(probs)
    conf = probs[pred_class]
    print(f"Prediction: {label_encoder.inverse_transform([pred_class])[0]}, Confidence: {conf:.2f}")
    
model.save_model("bn_model_13TypesXGB.json")  # or .bin

Prediction: gatewayping, Confidence: 0.99
Prediction: normal, Confidence: 1.00
Prediction: normal, Confidence: 1.00
Prediction: normal, Confidence: 1.00
Prediction: normal, Confidence: 1.00
